# Smart Health Testing — Hannum (full automatic pipeline)

End-to-end automatic inference on the **Hannum Smart Health** holdout (volunteers in Dataset100
`imagesTs`), then `eval_util.process_folders` for MD / FA / Dice / Hausdorff vs GT — same engine
as the inference and inter-reader notebooks.

**Pipeline** (per case, from the FULL native-spacing image):
1. **YOLO crop** on the full avg image → whole-heart square box.
2. **Crop + resize to 256** + per-channel normalization → `_0000.._0003`.
3. **nnUNet LV** (Dataset100, ResEnc-M, DA5-100ep) — **5-fold ensemble** → LV mask. *(shared)*
4. **Insertion points** — run once per method in `IP_METHODS` and compare:
   - `yolo_1c`  — YOLO RVIP, single contrast `(0,0,0)`
   - `yolo_mc`  — YOLO RVIP, multi-contrast `(0,1,2)` (YOLO is 3-channel)
   - `nnunet_4c`— *(enable once trained)* nnUNet IP, **all 4 contrasts** — the reviewer's
     "overkill" model; low cDTI contrast on a tiny target may need the extra capacity.
5. **Combine** LV(=1) + IPs(=2/3) and **`process_folders`** per method → one `.xlsx` each.

In [ ]:
import os
from datetime import datetime

# ============================ nnUNet LV model (Dataset100) ============================
dataset_id_lv = 100
trainer   = 'nnUNetTrainerDA5_100epochs'
# trainer/plans/folds below apply to the LV model AND any nnUNet IP method (all ResEnc-M).
plans     = 'nnUNetResEncUNetMPlans'          # ResEnc-M -- used for EVERY nnUNet predict
lv_config = '2d'
folds     = '0 1 2 3 4'                        # 5-fold ensemble (needs folds 0..4 trained)

# ============================ YOLO RVIP weights ============================
# ---- YOLO models selected by their TRAINING-dataset ID (tagged run folders from train_yolo_models s9) ----
crop_yolo_id = 110    # YOLO crop dataset ID  -> runs/crop_D110
rvip_yolo_id = 105    # YOLO RVIP dataset ID  -> runs/rvip_1c_D105 / runs/rvip_mc_D105
YOLO_RUNS = '/home/sastocke/nnUNet/yolo_datasets/runs'
YOLO_CROP_WEIGHTS        = f'{YOLO_RUNS}/crop_D{crop_yolo_id}/weights/best.pt'
YOLO_RVIP_WEIGHTS_SINGLE = f'{YOLO_RUNS}/rvip_1c_D{rvip_yolo_id}/weights/best.pt'   # 1-contrast
YOLO_RVIP_WEIGHTS_MULTI  = f'{YOLO_RUNS}/rvip_mc_D{rvip_yolo_id}/weights/best.pt'   # multi-contrast
CROP_CHANNELS     = (0, 0, 0)
RVIP_SINGLE_CLASS = True
RVIP_FLIP_ASSIGN  = False          # anterior = higher; flip if a GT check shows it reversed
RVIP_CONF         = 0.01           # low: RVIP models are weak; single-class takes top-2 regardless.
                                   # The 0.25 default drops real low-confidence points -> all-misses.

# ============================ IP methods to run & compare ============================
# YOLO is 3-channel RGB, so its "multi-contrast" is (0,1,2). The true 4-contrast detector is the
# nnUNet IP model (nnUNet reads all 4 channels) -> enable the nnunet_4c entry once it is trained.
IP_METHODS = [
    {'tag': 'yolo_1c', 'kind': 'yolo', 'weights': YOLO_RVIP_WEIGHTS_SINGLE, 'channels': (0, 0, 0)},
    {'tag': 'yolo_mc', 'kind': 'yolo', 'weights': YOLO_RVIP_WEIGHTS_MULTI,  'channels': (0, 1, 2)},
    # {'tag': 'nnunet_4c', 'kind': 'nnunet', 'dataset': 105, 'config': '2d',
    #  'trainer': trainer, 'plans': plans, 'folds': folds},   # nnUNet IP, all 4 contrasts (enable when trained)
    # {'tag': 'nnunet_1c', 'kind': 'nnunet', 'dataset': <ID_1c>, ...},
]

# ============================ data ============================
import glob as _glob
_lv_raw = _glob.glob(f'/home/sastocke/nnUNet/nnUNet_raw/Dataset{dataset_id_lv:03d}_*')
assert _lv_raw, f'no nnUNet_raw/Dataset{dataset_id_lv:03d}_* folder found'
IMAGES_TS      = f'{_lv_raw[0]}/imagesTs'                               # LV dataset test split (by ID)
TEST_DATA_ROOT = '/home/sastocke/nnUNet/HannumSmartHealthTestData'      # same layout as prep source
main_folder    = os.path.dirname(TEST_DATA_ROOT)
annotator      = os.path.basename(TEST_DATA_ROOT)                       # prefix == folder name (single token)
assert '_' not in annotator, f"annotator '{annotator}' has underscores; rename the test folder to one token."

# ============================ outputs (LV shared; IPs per-method) ============================
# results path encodes crop + RVIP + LV IDs so any run is fully reconstructable:
run       = f'SmartHealthTestHannum_crop{crop_yolo_id}_rvip{rvip_yolo_id}_lv{dataset_id_lv}'
base_out  = f'/home/sastocke/nnUNet/inferenceTest_{run}'
output_crop_masks   = f'{base_out}/CropInference'
output_image_folder = f'{base_out}/processed256/imagesTr'
output_LV           = f'{base_out}/OutputLV'
gt_folder           = f'{base_out}/GT_Combined'
inspection_folder   = f'{base_out}/inspection'
log_file_path       = f'{base_out}/{run}.log'
for d in [output_crop_masks, output_image_folder, output_LV, gt_folder, inspection_folder]:
    os.makedirs(d, exist_ok=True)

def clear_dirs(dirs):
    for d in dirs:
        if os.path.isdir(d):
            for f in os.listdir(d):
                p = os.path.join(d, f)
                if os.path.isfile(p) or os.path.islink(p):
                    os.unlink(p)
clear_dirs([output_crop_masks, output_image_folder, output_LV, gt_folder])


In [2]:
import glob, re
import numpy as np
import nibabel as nib
import cv2
import pandas as pd
import matplotlib.pyplot as plt

import yolo_pipeline as yp
from eval_util import process_folders, dice_score_original

os.environ["nnUNet_raw"]          = "/home/sastocke/nnUNet/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/home/sastocke/nnUNet/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = "/home/sastocke/nnUNet/nnUNet_results"

interpolation_size = 256

def normalize_avg_data(x):        return (x - np.min(x)) / (np.max(x) - np.min(x))
def normalize_mean_diff_data(x):  return (x - 0) / 4.0
def normalize_eigenvector_data(x):return x / np.sqrt(2)   # X+Y capped at sqrt(2) (unit eigenvector)

def make_square_crop(mask, target_size=256):
    coords = np.argwhere(mask > 0)
    x_min, y_min = coords.min(axis=0); x_max, y_max = coords.max(axis=0) + 1
    side = max(x_max - x_min, y_max - y_min)
    cx, cy = (x_min + x_max) // 2, (y_min + y_max) // 2
    x_min = max(0, cx - side // 2); x_max = x_min + side
    y_min = max(0, cy - side // 2); y_max = y_min + side
    return x_min, x_max, y_min, y_max, side / target_size

def parse_case(fname):
    stem = re.sub(r'(_0000)?\.nii(\.gz)?$', '', fname)
    parts = stem.split('_'); i = parts.index('Volunteer'); j = parts.index('slice')
    return f'{parts[i]}_{parts[i+1]}', f'{parts[i+2]}_{parts[i+3]}_{parts[i+4]}', f'{int(parts[j+1]):03d}'

def case_id(volunteer, divo, slc):   return f'{annotator}_{volunteer}_{divo}_slice_{slc}'
def source_dir(volunteer, divo):     return os.path.join(TEST_DATA_ROOT, volunteer, 'Distortion_Corrected', divo)

cases = [parse_case(os.path.basename(f)) for f in sorted(glob.glob(os.path.join(IMAGES_TS, '*_0000.nii.gz')))]
print(f'{len(cases)} test cases from imagesTs'); [print('  ', c) for c in cases[:8]]


0 test cases from imagesTs


[]

## 1. Build combined GT (LV=1, IP1=2, IP2=3) from `06_Segmentation_Masks_CI` *(shared)*

In [3]:
def combine_gt_mask(mask3):
    out = np.zeros(mask3.shape[:2], dtype=np.uint8)
    out[mask3[:, :, 0] > 0] = 1; out[mask3[:, :, 1] > 0] = 2; out[mask3[:, :, 2] > 0] = 3
    return out

built = 0
for volunteer, divo, slc in cases:
    gt_src = os.path.join(source_dir(volunteer, divo), '06_Segmentation_Masks_CI', f'Cropped_Segmentation_Slice_{slc}.nii')
    if not os.path.exists(gt_src):
        print('missing GT mask:', gt_src); continue
    m = nib.load(gt_src)
    nib.save(nib.Nifti1Image(combine_gt_mask(m.get_fdata()), m.affine),
             os.path.join(gt_folder, case_id(volunteer, divo, slc) + '.nii.gz'))
    built += 1
print(f'built {built} combined GT masks -> {gt_folder}')


built 0 combined GT masks -> /home/sastocke/nnUNet/inferenceTest_SmartHealthTestHannum_crop111_rvip107_lv106/GT_Combined


## 2. YOLO crop → crop + resize to 256 + normalize *(shared)*

In [4]:
crop_model = yp.load_model(YOLO_CROP_WEIGHTS)
for volunteer, divo, slc in cases:
    img_folder = os.path.join(source_dir(volunteer, divo), '03_Segmentation_Images')
    avg_f = os.path.join(img_folder, f'Average_Diffusion_Weighted_Image_Slice_{slc}.nii')
    md_f  = os.path.join(img_folder, f'Mean_Diffusivty_Image_Slice_{slc}.nii')
    eig_f = os.path.join(img_folder, f'Primary_Eigenvector_Image_Slice_{slc}.nii')
    fa_f  = os.path.join(img_folder, f'Fractional_Anisotropy_Image_Slice_{slc}.nii')
    if not all(os.path.exists(p) for p in [avg_f, md_f, eig_f, fa_f]):
        print('missing modality for', volunteer, divo, slc); continue
    avg_img = nib.load(avg_f); avg = avg_img.get_fdata()
    md_ = nib.load(md_f).get_fdata(); eig = nib.load(eig_f).get_fdata(); fa = nib.load(fa_f).get_fdata()

    box = yp.detect_crop_box(crop_model, avg)
    if box is None:
        print('no crop detected -> skipping', volunteer, divo, slc); continue
    crop_mask = yp.crop_box_to_mask(box, avg.shape)
    cid = case_id(volunteer, divo, slc)
    nib.save(nib.Nifti1Image(crop_mask.astype(np.uint8), avg_img.affine), os.path.join(output_crop_masks, cid + '.nii.gz'))

    x0, x1, y0, y1, _ = make_square_crop(crop_mask)
    avg_c = normalize_avg_data(      cv2.resize(avg[x0:x1, y0:y1],                         (256,256), interpolation=cv2.INTER_LINEAR))
    md_c  = normalize_mean_diff_data(cv2.resize(md_[x0:x1, y0:y1],                         (256,256), interpolation=cv2.INTER_LINEAR))
    eig_c = normalize_eigenvector_data(cv2.resize(eig[x0:x1, y0:y1, 0] + eig[x0:x1, y0:y1, 1], (256,256), interpolation=cv2.INTER_LINEAR))
    fa_c  =                          cv2.resize(fa[x0:x1, y0:y1],                          (256,256), interpolation=cv2.INTER_LINEAR)
    aff = np.eye(4)
    for ch, arr in zip(range(4), [avg_c, md_c, eig_c, fa_c]):
        nib.save(nib.Nifti1Image(arr, aff), os.path.join(output_image_folder, f'{cid}_{ch:04d}.nii.gz'))
    print('cropped+resized', cid)
print('crop + resize done ->', output_image_folder)


crop + resize done -> /home/sastocke/nnUNet/inferenceTest_SmartHealthTestHannum_crop111_rvip107_lv106/processed256/imagesTr


## 3. nnUNet LV segmentation — 5-fold ensemble *(shared)*

In [5]:
cmd = (f"nnUNetv2_predict -i {output_image_folder} -o {output_LV} "
       f"-d {dataset_id_lv} -c {lv_config} -tr {trainer} -p {plans} -f {folds}")
print(cmd)
os.system(cmd)


nnUNetv2_predict -i /home/sastocke/nnUNet/inferenceTest_SmartHealthTestHannum_crop111_rvip107_lv106/processed256/imagesTr -o /home/sastocke/nnUNet/inferenceTest_SmartHealthTestHannum_crop111_rvip107_lv106/OutputLV -d 106 -c 2d -tr nnUNetTrainerDA5_100epochs -p nnUNetResEncUNetMPlans -f 0 1 2 3 4

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################



Traceback (most recent call last):
  File "/home/sastocke/.conda/envs/nnunet/bin/nnUNetv2_predict", line 6, in <module>
    sys.exit(predict_entry_point())
             ^^^^^^^^^^^^^^^^^^^^^
  File "/home/sastocke/nnUNet/nnunetv2/inference/predict_from_raw_data.py", line 861, in predict_entry_point
    predictor.initialize_from_trained_model_folder(
  File "/home/sastocke/nnUNet/nnunetv2/inference/predict_from_raw_data.py", line 74, in initialize_from_trained_model_folder
    dataset_json = load_json(join(model_training_output_dir, 'dataset.json'))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sastocke/.conda/envs/nnunet/lib/python3.11/site-packages/batchgenerators/utilities/file_and_folder_operations.py", line 103, in load_json
    with open(file, 'r') as f:
         ^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/home/sastocke/nnUNet/nnUNet_results/Dataset106_HannumDirVsAvgIPs/nnUNetTrainerDA5_100epochs__nnUNetRe

256

## 4. Insertion points — run each method in `IP_METHODS` and compare

Each method: detect IPs → combine with the shared LV → `process_folders` → its own `.xlsx`.
LV Dice / MD / FA are identical across methods (LV is shared); the **Hausdorff Label 2/3**
columns are what distinguish the IP methods.

In [6]:
def detect_ips_yolo(m, ip_dir):
    model = yp.load_model(m['weights'])
    for volunteer, divo, slc in cases:
        cid = case_id(volunteer, divo, slc)
        img0 = os.path.join(output_image_folder, f'{cid}_0000.nii.gz')
        if not os.path.exists(img0):
            continue
        pts = yp.detect_rvips_from_path(model, img0, channels=m['channels'], single_class=RVIP_SINGLE_CLASS,
                                        flip_assignment=RVIP_FLIP_ASSIGN, conf_thresh=RVIP_CONF)
        ipm = yp.rvips_to_mask(pts, (256, 256), anterior_label=2, inferior_label=3)
        nib.save(nib.Nifti1Image(ipm.astype(np.uint8), np.eye(4)), os.path.join(ip_dir, cid + '.nii.gz'))

def detect_ips_nnunet(m, ip_dir):
    cmd = (f"nnUNetv2_predict -i {output_image_folder} -o {ip_dir} -d {m['dataset']} "
           f"-c {m['config']} -tr {m['trainer']} -p {m['plans']} -f {m['folds']}")
    print(cmd); os.system(cmd)   # nnUNet IP labels IP1=1, IP2=2 -> remapped to 2/3 in combine()

def combine(kind, ip_dir, final_dir):
    for f in os.listdir(output_LV):
        if not f.endswith('.nii.gz'):
            continue
        lv_img = nib.load(os.path.join(output_LV, f)); lv = lv_img.get_fdata().astype(np.int32)
        out = lv.copy(); out[lv == 1] = 1
        ip_path = os.path.join(ip_dir, f)
        if os.path.exists(ip_path):
            ip = nib.load(ip_path).get_fdata().astype(np.int32)
            a, b = (2, 3) if kind == 'yolo' else (1, 2)   # yolo already 2/3; nnUNet 1/2 -> 2/3
            out[ip == a] = 2; out[ip == b] = 3
        nib.save(nib.Nifti1Image(out, lv_img.affine, lv_img.header), os.path.join(final_dir, f))

def run_ip_method(m):
    tag = m['tag']
    ip_dir    = f'{base_out}/OutputIPs_{tag}'
    final_dir = f'{base_out}/Output_Final_Combined_{tag}'
    for d in (ip_dir, final_dir):
        os.makedirs(d, exist_ok=True)
    clear_dirs([ip_dir, final_dir])
    (detect_ips_yolo if m['kind'] == 'yolo' else detect_ips_nnunet)(m, ip_dir)
    combine(m['kind'], ip_dir, final_dir)
    md_data, mm = [], []
    process_folders(final_dir, gt_folder, main_folder, md_data, mm, None, annotator, output_crop_masks)
    df = pd.DataFrame(md_data)
    xlsx = f'{base_out}/{run}_{tag}.xlsx'; df.to_excel(xlsx, index=False); df.to_csv(xlsx.replace('.xlsx', '.csv'), index=False)
    print(f'[{tag}] {len(df)} cases -> {xlsx}')
    return tag, df

results = {}
for m in IP_METHODS:
    print('=' * 30, m['tag'], '=' * 30)
    tag, df = run_ip_method(m)
    results[tag] = df


============================== yolo_1c ==============================


FileNotFoundError: [Errno 2] No such file or directory: '/home/sastocke/nnUNet/yolo_datasets/runs/rvip_1c_D107/weights/best.pt'

## 5. Comparison summary across IP methods

In [ ]:
cols = ['Dice Score Original Label 1', 'Hausdorff Distance Label 2', 'Hausdorff Distance Label 3',
        'GT_Median_MD', 'Pred_median_MD', 'GT_Median_FA', 'Pred_median_FA', 'Avg. HD Epi', 'Avg. HD Endo']
summary = pd.DataFrame({tag: df[cols].mean() for tag, df in results.items()})
print(summary)
summary.to_excel(f'{base_out}/{run}_IPmethod_comparison.xlsx'); summary.to_csv(f'{base_out}/{run}_IPmethod_comparison.csv')
# a Hausdorff of ~1250 = the eval failsafe for a MISSED point (1000 x 1.25); a good detector is single-digit px.
